# CineData Analytics — Silver → Gold
Star Schema para consumo de BI: 1 fato, 5 dimensões e 3 tabelas-ponte (N:N).
Chaves substitutas via `xxhash64` (BIGINT **determinístico**: a mesma chave natural gera a mesma SK a cada execução, então o job pode rodar várias vezes sem quebrar FKs).

In [ ]:
from pyspark.sql import functions as F

spark.sql("CREATE DATABASE IF NOT EXISTS gold")

def salva_gold(df, tabela):
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    print(f"{tabela}: {spark.table(tabela).count()} linhas")

def sk(*cols, prefixo):
    """Surrogate key BIGINT por hash determinístico (prefixo evita colisão entre dimensões)."""
    return F.xxhash64(F.lit(prefixo), *[F.col(c) for c in cols])

s_info  = spark.table("silver.tb_info_filmes")
s_fin   = spark.table("silver.tb_financeiro_filmes")
s_met   = spark.table("silver.tb_metricas_engajamento")
s_rev   = spark.table("silver.tb_avaliacoes_usuarios")
s_gen   = spark.table("silver.tb_generos")
s_pe    = spark.table("silver.tb_pessoas_empresas")

## Dimensões

In [ ]:
# dim_movies
dim_movies = (s_info.dropDuplicates(["id_filme"])
    .select(sk("id_filme", prefixo="movie").alias("sk_movie_id"),
            F.col("id_filme").cast("string"), "titulo",
            F.col("data_lancamento").cast("date"), F.col("ano_lancamento").cast("int"),
            F.col("duracao_minutos").cast("int"), "idioma_original", "status_filme", "sinopse"))
salva_gold(dim_movies, "gold.dim_movies")

# dim_genres
dim_genres = (s_gen.select(F.col("genero").alias("nome_genero")).distinct()
    .select(sk("nome_genero", prefixo="genre").alias("sk_genre_id"), "nome_genero"))
salva_gold(dim_genres, "gold.dim_genres")

# dim_people (pessoas físicas: Ator, Diretor, Roteirista). Grão = nome + tipo.
dim_people = (s_pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(F.col("nome").alias("nome_pessoa"), F.col("tipo_entidade").alias("tipo_pessoa")).distinct()
    .select(sk("nome_pessoa", "tipo_pessoa", prefixo="person").alias("sk_person_id"), "nome_pessoa", "tipo_pessoa"))
salva_gold(dim_people, "gold.dim_people")

# dim_companies
dim_companies = (s_pe.filter("tipo_entidade = 'Produtora'").select(F.col("nome").alias("nome_produtora")).distinct()
    .select(sk("nome_produtora", prefixo="company").alias("sk_company_id"), "nome_produtora"))
salva_gold(dim_companies, "gold.dim_companies")

# dim_reviews: avaliações resumidas por filme (1 linha por filme avaliado)
dim_reviews = (s_rev.groupBy("id_filme")
    .agg(F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
         F.round(F.avg("nota_usuario"), 2).cast("double").alias("nota_media_usuarios"))
    .join(dim_movies.select("sk_movie_id", F.col("id_filme").alias("_id")), F.col("id_filme") == F.col("_id"), "inner")
    .select(sk("id_filme", prefixo="review").alias("sk_review_id"), "sk_movie_id",
            "qtd_avaliacoes_usuarios", "nota_media_usuarios"))
salva_gold(dim_reviews, "gold.dim_reviews")

## Fato

In [ ]:
# Grão: 1 registro por filme. Silver financeiro e métricas já são únicas por id_filme,
# então os LEFT JOINs a partir da dim_movies não duplicam o grão.
fact = (dim_movies.select("sk_movie_id", "id_filme")
    .join(s_fin, "id_filme", "left")
    .join(s_met, "id_filme", "left")
    .select("sk_movie_id",
            F.col("orcamento_usd").cast("decimal(18,2)"), F.col("receita_usd").cast("decimal(18,2)"),
            F.col("lucro_usd").cast("decimal(18,2)"),
            F.col("orcamento_brl").cast("decimal(18,2)"), F.col("receita_brl").cast("decimal(18,2)"),
            F.col("lucro_brl").cast("decimal(18,2)"),
            F.col("popularidade").cast("double"), F.col("nota_media_tmdb").cast("double"),
            F.col("qtd_votos_tmdb").cast("int"), F.col("nota_media_imdb").cast("double"),
            F.col("qtd_votos_imdb").cast("int"))
    .dropDuplicates(["sk_movie_id"]))   # garantia extra do grão
salva_gold(fact, "gold.fact_movies_performance")
assert fact.count() == fact.select("sk_movie_id").distinct().count(), "Grão da fato duplicado!"

## Tabelas-ponte

In [ ]:
m = dim_movies.select("sk_movie_id", F.col("id_filme").alias("_id"))

bridge_genre = (s_gen.join(m, s_gen.id_filme == m._id)
    .select("sk_movie_id", sk("genero", prefixo="genre").alias("sk_genre_id")).distinct())
salva_gold(bridge_genre, "gold.bridge_movie_genre")

bridge_person = (s_pe.filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .join(m, s_pe.id_filme == m._id)
    .select("sk_movie_id", sk("nome", "tipo_entidade", prefixo="person").alias("sk_person_id")).distinct())
salva_gold(bridge_person, "gold.bridge_movie_person")

bridge_company = (s_pe.filter("tipo_entidade = 'Produtora'")
    .join(m, s_pe.id_filme == m._id)
    .select("sk_movie_id", sk("nome", prefixo="company").alias("sk_company_id")).distinct())
salva_gold(bridge_company, "gold.bridge_movie_company")

# Desafio de Analytics

### 1. Receita total (R$) de todos os filmes

In [ ]:
display(spark.sql("""
SELECT CAST(SUM(receita_brl) AS DECIMAL(24,2)) AS receita_total_brl
FROM gold.fact_movies_performance
"""))

### 2. Top 5 filmes por popularidade

In [ ]:
display(spark.sql("""
SELECT m.titulo, f.popularidade
FROM gold.fact_movies_performance f
JOIN gold.dim_movies m USING (sk_movie_id)
WHERE f.popularidade IS NOT NULL
ORDER BY f.popularidade DESC
LIMIT 5
"""))

### 3. Quantidade de filmes por gênero

In [ ]:
display(spark.sql("""
SELECT g.nome_genero, COUNT(DISTINCT b.sk_movie_id) AS qtd_filmes
FROM gold.bridge_movie_genre b
JOIN gold.dim_genres g USING (sk_genre_id)
GROUP BY g.nome_genero
ORDER BY qtd_filmes DESC
"""))

### 4. Top 10 filmes por receita, com RANK()

In [ ]:
display(spark.sql("""
SELECT m.titulo, f.receita_usd, f.receita_brl,
       RANK() OVER (ORDER BY f.receita_usd DESC) AS posicao_ranking
FROM gold.fact_movies_performance f
JOIN gold.dim_movies m USING (sk_movie_id)
WHERE f.receita_usd IS NOT NULL
QUALIFY posicao_ranking <= 10
ORDER BY posicao_ranking
"""))

### Data de referência para os recortes de 2 e 5 anos
Maior data de lançamento **já realizada**: só filmes com status "Lançado" e data <= hoje (ignora datas futuras/não lançados).

In [ ]:
ref = spark.sql("""
SELECT MAX(data_lancamento) AS d FROM gold.dim_movies
WHERE status_filme = 'Lançado' AND data_lancamento <= current_date()
""").first().d
print("Data limite superior:", ref)

### 5. Ator com mais participações em filmes lançados nos últimos 2 anos

In [ ]:
display(spark.sql(f"""
SELECT p.nome_pessoa AS ator, COUNT(DISTINCT m.sk_movie_id) AS qtd_participacoes
FROM gold.dim_movies m
JOIN gold.bridge_movie_person b USING (sk_movie_id)
JOIN gold.dim_people p USING (sk_person_id)
WHERE p.tipo_pessoa = 'Ator'
  AND m.status_filme = 'Lançado'
  AND m.data_lancamento >  add_months(DATE'{ref}', -24)
  AND m.data_lancamento <= DATE'{ref}'
GROUP BY p.nome_pessoa
ORDER BY qtd_participacoes DESC, ator
LIMIT 10
"""))   # a 1ª linha é a resposta; as demais mostram empates/contexto

### 6. Produtora com maior lucro nos últimos 5 anos

In [ ]:
display(spark.sql(f"""
SELECT c.nome_produtora,
       SUM(f.lucro_usd) AS lucro_total_usd,
       SUM(f.lucro_brl) AS lucro_total_brl
FROM gold.dim_movies m
JOIN gold.fact_movies_performance f USING (sk_movie_id)
JOIN gold.bridge_movie_company b USING (sk_movie_id)
JOIN gold.dim_companies c USING (sk_company_id)
WHERE m.status_filme = 'Lançado'
  AND m.data_lancamento >  add_months(DATE'{ref}', -60)
  AND m.data_lancamento <= DATE'{ref}'
GROUP BY c.nome_produtora
ORDER BY lucro_total_usd DESC NULLS LAST
LIMIT 10
"""))